# FlyPose-SAR — Fine-tuning YOLO Multi-Palette Thermal (Fase 2b)

## Prerequisiti (Data):
- `dataset-sar-thermal-multipalette` — dataset generato in locale con generate_thermal_multipalette.py
- `hit-uav` — termico reale zenitale (box only)
- `flypose-fase1-large-weights` — pesi Large SAR Fase 1

## Obiettivo:
Allenare un modello YOLO11-Pose in grado di rilevare persone
su immagini termiche con qualsiasi palette (White Hot, Black Hot,
Iron Red, Rainbow 1, Hot Iron) + termico reale HIT-UAV.

## Cella 1 — Installazione

In [ ]:
!pip install ultralytics -q
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

## Cella 2 — Trova dataset e pesi

In [ ]:
from pathlib import Path

KAGGLE_INPUT   = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')

# ---- dataset multipalette ----
multipalette_path = None
for d in KAGGLE_INPUT.rglob('dataset_sar_thermal_multipalette'):
    if d.is_dir():
        multipalette_path = d
        break

# ---- HIT-UAV ----
hituav_path = None
for d in KAGGLE_INPUT.rglob('hit-uav'):
    if d.is_dir() and (d / 'images').exists():
        hituav_path = d
        break
if hituav_path is None:
    for d in KAGGLE_INPUT.rglob('images'):
        root = d.parent
        if 'hit' in str(root).lower() and (root / 'labels' / 'train').exists():
            hituav_path = root
            break

# ---- pesi Fase 1 ----
fase1_weights = None
for p in KAGGLE_INPUT.rglob('*.pt'):
    if 'large' in p.name.lower() or 'best' in p.name.lower():
        fase1_weights = p
        break
if fase1_weights is None:
    for p in KAGGLE_INPUT.rglob('*.pt'):
        fase1_weights = p
        break

print(f'Multipalette   : {multipalette_path}')
print(f'HIT-UAV        : {hituav_path}')
print(f'Pesi Fase 1    : {fase1_weights}')

PALETTES = ['white_hot', 'black_hot', 'iron_red', 'rainbow1', 'hot_iron']
if multipalette_path:
    for p in PALETTES:
        n = len(list((multipalette_path / p / 'images' / 'train').glob('*.jpg')))
        print(f'  {p:12s}: {n} immagini train')

## Cella 3 — Costruisce dataset misto multi-palette

In [ ]:
import shutil, random
from pathlib import Path

# ---- PARAMETRI ----
SUBSAMPLE_PER_PALETTE = 1    # usa tutti i frame (gia' subsampati in generazione)
HITUAV_PERSON_CLASS   = 0
N_KPT                 = 17
VAL_SPLIT             = 0.05 # 5% come val
SEED                  = 42
# -------------------

random.seed(SEED)
mix_root = KAGGLE_WORKING / 'dataset_thermal_multipalette_mix'
for split in ['train', 'val']:
    (mix_root / 'images' / split).mkdir(parents=True, exist_ok=True)
    (mix_root / 'labels' / split).mkdir(parents=True, exist_ok=True)

def copy_pair(src_img, src_lbl, dst_img_dir, dst_lbl_dir, stem_prefix=''):
    dst_stem = stem_prefix + src_img.stem
    shutil.copy(src_img, dst_img_dir / (dst_stem + src_img.suffix))
    if src_lbl.exists():
        shutil.copy(src_lbl, dst_lbl_dir / (dst_stem + '.txt'))

# ---- 1. Sintetico multi-palette ----
total_syn = 0
for palette in PALETTES:
    pal_img_dir = multipalette_path / palette / 'images' / 'train'
    pal_lbl_dir = multipalette_path / palette / 'labels' / 'train'
    imgs = sorted(pal_img_dir.glob('*.jpg'))
    random.shuffle(imgs)
    n_val = int(len(imgs) * VAL_SPLIT)
    val_imgs, train_imgs = imgs[:n_val], imgs[n_val:]
    for p in train_imgs:
        copy_pair(p, pal_lbl_dir / (p.stem + '.txt'),
                  mix_root / 'images' / 'train',
                  mix_root / 'labels' / 'train',
                  stem_prefix=f'{palette}_')
    for p in val_imgs:
        copy_pair(p, pal_lbl_dir / (p.stem + '.txt'),
                  mix_root / 'images' / 'val',
                  mix_root / 'labels' / 'val',
                  stem_prefix=f'{palette}_')
    total_syn += len(train_imgs)
    print(f'  {palette:12s}: {len(train_imgs)} train + {len(val_imgs)} val')

# ---- 2. HIT-UAV reale (box only, kpts=0) ----
kpt_zeros = ' '.join(['0 0 0'] * N_KPT)
n_hit = 0
hit_img_dir = hituav_path / 'images' / 'train'
hit_lbl_dir = hituav_path / 'labels' / 'train'
for lbl_path in sorted(hit_lbl_dir.glob('*.txt')):
    lines_in  = lbl_path.read_text().strip().splitlines()
    lines_out = []
    for line in lines_in:
        parts = line.split()
        if not parts or int(float(parts[0])) != HITUAV_PERSON_CLASS:
            continue
        lines_out.append('0 ' + ' '.join(parts[1:5]) + ' ' + kpt_zeros)
    if not lines_out:
        continue
    img_path = None
    for ext in ['.jpg', '.jpeg', '.png']:
        c = hit_img_dir / (lbl_path.stem + ext)
        if c.exists():
            img_path = c
            break
    if img_path is None:
        continue
    dst_stem = 'hit_' + lbl_path.stem
    shutil.copy(img_path, mix_root / 'images' / 'train' / (dst_stem + img_path.suffix))
    (mix_root / 'labels' / 'train' / (dst_stem + '.txt')).write_text('\n'.join(lines_out))
    n_hit += 1

n_train = len(list((mix_root / 'images' / 'train').glob('*.*')))
n_val   = len(list((mix_root / 'images' / 'val').glob('*.*')))
print(f'\nDataset finale:')
print(f'  Train: {n_train} ({total_syn} sintetici multi-palette + {n_hit} HIT-UAV reali)')
print(f'  Val  : {n_val}')

## Cella 4 — Scrive data.yaml

In [ ]:
import yaml

data_yaml = {
    'path'      : str(mix_root),
    'train'     : 'images/train',
    'val'       : 'images/val',
    'nc'        : 1,
    'names'     : ['person'],
    'kpt_shape' : [17, 3],
}
yaml_path = mix_root / 'data_multipalette.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)
print(f'[OK] {yaml_path}')
print(open(yaml_path).read())

## Cella 5 — Fine-tuning YOLO11-Pose

In [ ]:
from ultralytics import YOLO

# ---- PARAMETRI ----
MODEL_SIZE = 'large'
EPOCHS     = 50
BATCH      = 16
IMGSZ      = 640
LR0        = 0.0002
LRF        = 0.01
# -------------------

model = YOLO(str(fase1_weights))

results = model.train(
    data         = str(yaml_path),
    epochs       = EPOCHS,
    batch        = BATCH,
    imgsz        = IMGSZ,
    lr0          = LR0,
    lrf          = LRF,
    warmup_epochs= 3,
    cos_lr       = True,
    name         = f'flypose_thermal_multipalette_{MODEL_SIZE}',
    project      = str(KAGGLE_WORKING / 'runs_multipalette'),
    exist_ok     = True,
    device       = 0,
    workers      = 4,
    patience     = 20,
    save         = True,
    plots        = True,
)

print(f'[OK] Training completato')
print(f'Best weights: {results.save_dir}/weights/best.pt')

## Cella 6 — Valutazione formale

In [ ]:
from ultralytics import YOLO

best_pt = list((KAGGLE_WORKING / 'runs_multipalette' /
                f'flypose_thermal_multipalette_{MODEL_SIZE}' /
                'weights').glob('best.pt'))[0]
model   = YOLO(str(best_pt))
metrics = model.val(data=str(yaml_path), imgsz=IMGSZ, batch=BATCH, device=0)

print(f'\n=== Risultati Fase 2b — {MODEL_SIZE.upper()} Multi-Palette Thermal ===')
print(f'Box  mAP@0.5  : {metrics.box.map50:.4f}')
print(f'Pose mAP@0.5  : {metrics.pose.map50:.4f}')
print(f'Precision     : {metrics.box.mp:.4f}')
print(f'Recall        : {metrics.box.mr:.4f}')
print(f'\nConfronto:')
print(f'  Fase 1 RGB              : Box=0.5961  Pose=0.3852')
print(f'  Fase 2 Thermal sintetico: Box=0.7994  Pose=0.5505')

## Cella 7 — Zip e download

In [ ]:
import zipfile

run_dir  = KAGGLE_WORKING / 'runs_multipalette' / f'flypose_thermal_multipalette_{MODEL_SIZE}'
zip_path = KAGGLE_WORKING / f'flypose_multipalette_{MODEL_SIZE}.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for pt in (run_dir / 'weights').glob('*.pt'):
        zf.write(pt, f'weights/{pt.name}')
        print(f'  [+] {pt.name}')
    for img in run_dir.glob('*.png'):
        zf.write(img, f'plots/{img.name}')
        print(f'  [+] {img.name}')
    csv = run_dir / 'results.csv'
    if csv.exists():
        zf.write(csv, 'results.csv')
        print(f'  [+] results.csv')

print(f'\n[OK] {zip_path} — {zip_path.stat().st_size/1e6:.1f} MB')
print('Scarica da Output (pannello destro)')